In [151]:
from datetime import timedelta, datetime

try:
    from pyspark.sql import functions as F
    from pyspark.sql.functions import year, col, length, lit, ltrim, month
except Exception:
    F = None
    year = col = length = lit = ltrim = month = None

from pandas import ExcelWriter, read_excel
import datetime as dt
import pandas as pd
import numpy as np
import warnings
import copy
import time
import re
import shutil

In [ ]:
import os

# SQL warehouse credentials for local table extraction only.
creds = {
                "server_hostname": "",
    "http_path":"",
    "access_token":"",
        }

db_host = creds.get("server_hostname", "").strip() or os.getenv("DATABRICKS_HOST", "").strip()
if db_host and not db_host.startswith("http"):
    db_host = f"https://{db_host}"

db_http_path = creds.get("http_path", "").strip() or os.getenv("DATABRICKS_HTTP_PATH", "").strip()
db_token = (
    creds.get("access_token", "").strip()
    or creds.get("token", "").strip()
    or os.getenv("DATABRICKS_TOKEN", "").strip()
)

missing = [
    name
    for name, value in [
        ("server_hostname / DATABRICKS_HOST", db_host),
        ("http_path / DATABRICKS_HTTP_PATH", db_http_path),
        ("access_token|token / DATABRICKS_TOKEN", db_token),
    ]
    if not value
]
if missing:
    raise ValueError("Missing Databricks SQL credentials: " + ", ".join(missing))

# Local widgets shim for compatibility with downstream cells if needed.
if "dbutils" not in globals():
    class _LocalWidgets:
        _env_map = {
            "historic": "DOP_HISTORIC",
            "curr_date": "DOP_CURR_DATE",
        }

        @classmethod
        def get(cls, name):
            env_name = cls._env_map.get(name, name.upper())
            return os.getenv(env_name, "")

    class _LocalNotebook:
        @staticmethod
        def exit(message):
            raise RuntimeError(message)

    class _LocalDbUtils:
        widgets = _LocalWidgets()
        notebook = _LocalNotebook()

    dbutils = _LocalDbUtils()

print("Databricks SQL bootstrap ready")

Databricks SQL bootstrap ready


In [153]:
if "spark" in globals():
    spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")           # to can read files from datalake
warnings.simplefilter(action='ignore', category=Warning)                        # don't show warnings code
pd.set_option('display.max_columns', None)

# Single run toggle: set to 'daily' or 'historic' (env var can override this value).
RUN_MODE = "daily "
RUN_MODE = os.getenv("DOP_RUN_MODE", RUN_MODE).strip().lower()
if RUN_MODE not in {"daily", "historic"}:
    raise ValueError(f"Invalid DOP_RUN_MODE={RUN_MODE}. Use 'daily' or 'historic'.")
run_output_dir = os.path.join("local_outputs", "mode_runs", RUN_MODE)

# Step output folder for audit snapshots.
step_outputs_dir = os.path.join(run_output_dir, "step_outputs")
os.makedirs(step_outputs_dir, exist_ok=True)
for f in os.listdir(step_outputs_dir):
    if f.lower().endswith(".csv"):
        os.remove(os.path.join(step_outputs_dir, f))

_STEP_COUNTER = 0

def save_step_df(df, step_name):
    global _STEP_COUNTER
    if not isinstance(df, pd.DataFrame):
        return
    _STEP_COUNTER += 1
    safe_name = re.sub(r"[^a-zA-Z0-9_\-]", "_", step_name.strip().lower())
    out_file = os.path.join(step_outputs_dir, f"{_STEP_COUNTER:02d}_{safe_name}.csv")
    df.to_csv(out_file, index=False)
    print(f"saved step output: {out_file} shape={df.shape}")

In [154]:
# Simple local extraction for the two source tables used in this notebook.
# Set DOP_LOCAL_EXTRACT_MAX_ROWS=0 (default) for full extraction within the date window.
max_rows = int(os.getenv("DOP_LOCAL_EXTRACT_MAX_ROWS", "0"))
batch_rows = int(os.getenv("DOP_LOCAL_EXTRACT_BATCH_ROWS", "10000"))
os.makedirs("local_outputs", exist_ok=True)

import requests
import time as _time
from datetime import date, timedelta

if "/warehouses/" not in db_http_path:
    raise ValueError("http_path must contain /warehouses/<warehouse_id>")

warehouse_id = db_http_path.split("/warehouses/")[-1].split("/")[0]

# Deterministic extraction window: always from today back 5 years (with env override support).
extract_today_date = os.getenv("DOP_CURR_DATE", date.today().strftime("%Y-%m-%d"))
extract_historic_start_date = os.getenv(
    "DOP_HISTORIC_START_DATE",
    (pd.to_datetime(extract_today_date, errors="coerce") - pd.DateOffset(years=5)).strftime("%Y-%m-%d"),
)

def run_sql_statement_to_pandas(statement):
    headers = {
        "Authorization": f"Bearer {db_token}",
        "Content-Type": "application/json",
    }
    payload = {
        "statement": statement,
        "warehouse_id": warehouse_id,
        "wait_timeout": "50s",
        "disposition": "INLINE",
        "format": "JSON_ARRAY",
    }

    submit = requests.post(f"{db_host}/api/2.0/sql/statements", headers=headers, json=payload, timeout=60)
    submit.raise_for_status()
    body = submit.json()

    state = body.get("status", {}).get("state")
    statement_id = body.get("statement_id")

    # Poll until statement finishes if it is still running.
    while state in {"PENDING", "RUNNING"}:
        _time.sleep(2)
        check = requests.get(f"{db_host}/api/2.0/sql/statements/{statement_id}", headers=headers, timeout=60)
        check.raise_for_status()
        body = check.json()
        state = body.get("status", {}).get("state")

    if state != "SUCCEEDED":
        err = body.get("status", {}).get("error", {})
        raise RuntimeError(f"SQL statement failed: {state} - {err}")

    cols = [c.get("name") for c in body.get("manifest", {}).get("schema", {}).get("columns", [])]
    rows = body.get("result", {}).get("data_array", [])
    return pd.DataFrame(rows, columns=cols)

def run_table_to_pandas(table_name, date_col=None):
    all_parts = []
    offset = 0
    total_rows = 0
    current_batch = max(500, batch_rows)
    unlimited_rows = max_rows <= 0

    where_clause = ""
    order_clause = ""
    if date_col:
        # End-inclusive for timestamp/date columns: include rows up to end-of-day.
        where_clause = (
            f" WHERE {date_col} >= DATE('{extract_historic_start_date}')"
            f" AND {date_col} < DATE_ADD(DATE('{extract_today_date}'), 1)"
        )
        # Deterministic pagination and newest-first retrieval.
        order_clause = f" ORDER BY {date_col} DESC"

    while unlimited_rows or (total_rows < max_rows):
        stmt = f"SELECT * FROM {table_name}{where_clause}{order_clause} LIMIT {current_batch} OFFSET {offset}"
        try:
            part = run_sql_statement_to_pandas(stmt)
        except RuntimeError as exc:
            # Reduce page size if a chunk still exceeds the inline payload cap.
            if "Inline byte limit exceeded" in str(exc) and current_batch > 500:
                current_batch = max(500, current_batch // 2)
                continue
            raise

        if part.empty:
            break

        all_parts.append(part)
        fetched = len(part)
        total_rows += fetched
        offset += fetched

        if fetched < current_batch:
            break

        if (not unlimited_rows) and (total_rows >= max_rows):
            break

    if not all_parts:
        return pd.DataFrame()
    out = pd.concat(all_parts, ignore_index=True)
    if (not unlimited_rows) and (len(out) > max_rows):
        out = out.iloc[:max_rows].copy()
    return out

extract_plan = [
    ("brewdat_uc_aurora_prod.slv_maz_masterdata_s4h.lfa1", os.path.join(run_output_dir, "dop_lfa1_extract.csv"), None),
    ("brewdat_uc_maz_prod.gld_maz_finance_account_payables.do_account_payables", os.path.join(run_output_dir, "dop_account_payables_extract.csv"), "posting_date"),
]

print(f"Date window for extraction: {extract_historic_start_date} to {extract_today_date}")
print(f"Row cap: {'unlimited' if max_rows <= 0 else max_rows}")

for table_name, out_path, date_col in extract_plan:
    try:
        pdf = run_table_to_pandas(table_name, date_col=date_col)
        pdf.to_csv(out_path, index=False)
        save_step_df(pdf, f"extract_{table_name.split('.')[-1]}")
        print("Saved:", out_path, pdf.shape)
    except Exception as exc:
        print(f"Failed to extract {table_name}: {exc}")

Date window for extraction: 2021-03-23 to 2026-03-23
Row cap: unlimited
saved step output: local_outputs\mode_runs\historic\step_outputs\01_extract_lfa1.csv shape=(4415, 207)
Saved: local_outputs\mode_runs\historic\dop_lfa1_extract.csv (4415, 207)
saved step output: local_outputs\mode_runs\historic\step_outputs\02_extract_do_account_payables.csv shape=(6566, 82)
Saved: local_outputs\mode_runs\historic\dop_account_payables_extract.csv (6566, 82)


In [155]:
import pkgutil
mods = sorted([m.name for m in pkgutil.iter_modules() if m.name.startswith("databricks")])
print(mods[:20])

['databricks']


## --------------------------------------- FUNCTIONS ---------------------------------------------

In [156]:
def get_duplicates_daily(df_hist, df_daily, last_ID, logics):
  
  for c in df_daily.columns: df_daily[c] = df_daily[c].astype(str)                      # formating all columns as strings
  for c in df_hist.columns: df_hist[c] = df_hist[c].astype(str)                         # formating all columns as strings
  
  duplicate_out = pd.DataFrame()
  countID = last_ID+1                                                                   # variable for the ID cases assignation
  df_daily['status'] = ''                                                                    
  for log_i in logics:                                                                  # for each logid
    logic_name = '|'.join(log_i)                                                        # the logic columns keys used

    if log_i == ['uuid']:                                                               # formating the column UUID
      uuid_logic = df_daily['uuid']!=''                                                      
      if df_daily[uuid_logic].empty: continue
      df_excluded = df_daily[~uuid_logic]                                               # excluding empty uuid to no exponential duplicates
      df_daily_c = df_daily[uuid_logic]
    else: df_daily_c = df_daily

    # ------------------------------------------------------------------- formating both dfs
    daily_indexes = list(df_daily_c.index)
    histo_indexes = list(df_hist.index)

    if df_daily_c.empty: df_daily_c['key'] = ''
    else:                df_daily_c['key'] = df_daily_c[log_i].apply('|'.join, axis=1).values # the key column is the concated of the values of the keys
     
    if df_hist.empty: df_hist['key'] = '' 
    else: df_hist['key'] = df_hist[log_i].apply('|'.join, axis=1).values      # the key column is the concated of the values of the keys

    df_hist['indexes_h'] = histo_indexes
    df_daily_c['indexes_d'] = daily_indexes                                     # saving the original index value for the merge

    df_daily_c['ID'] = [int(i) for i in range(countID, countID+len(df_daily_c))]  # assigned each ID number to the each case 
    df_hist['state'] = True

    daily_set = df_daily_c[['key', 'ID']]
    # -------------------------------------------------------------------
    daily_dup = pd.merge(daily_set, df_hist, on='key', how='left')    # merging the cases with the original duplicate set by key
    daily_dup['uuid'] = daily_dup['uuid'].astype(str)
    daily_dup = daily_dup[(daily_dup['state'] == True)]                  # the True values are the original values in the historical    
    daily_dup['status'] = 'Original'                                  # tag it as original cause appears first
    if daily_dup.shape[0] == 0: continue                              # haven't duplicates in historical

    ID_dups = daily_dup['ID'].unique()                                                         
    all_dups = df_daily_c[df_daily_c.ID.isin(ID_dups)]                    # from the daily data original, extract the ID found in historical
    all_dups['status'] = 'Duplicated'                                  # the daily found is the duplicate 

    indexes_used = list(all_dups.indexes_d)          
    df_daily_c = df_daily_c.drop(indexes_used, axis=0)                # drop the indexes used in the taged to the next logic dont used them
    duplicates = pd.concat([daily_dup, all_dups])                     # concat the originals and the duplicates                          
    duplicates['Logic'] = logic_name                                               
    
    if duplicates.empty: continue

    duplicate_out = pd.concat([duplicate_out, duplicates])            # save it in the dataframe main

    if log_i == ['uuid']: df_daily = pd.concat([df_excluded, df_daily_c])               # adding the excluded empty uuid at the begin
    countID += len(df_daily)        

  return duplicate_out

In [157]:
def get_duplicates_hist(df, logics):
  
    duplicate_out = pd.DataFrame(columns=df.columns)                                      # dataframe for the originals with duplicates
    duplicate_out['ID'] = ''
    original_df = copy.deepcopy(df)
    df = df.reset_index(drop=True)    
    for c in df.columns: 
        df[c] = df[c].astype(str)                                        # formating all columns as strings
        df[c] = df[c].str.strip()                                        # eliminating empty spaces

    df['status'] = ''                                                                    
    countID = 1                                                                           # variable for the ID cases assignation
    for log_i in logics:                                                                  # for each logid
        
        logic_name = '|'.join(log_i)                                                      # the logic columns keys used

        if log_i == ['uuid']:                                                             # formating the column UUID
            uuid_logic = df['uuid']==''                                                      
            df_excluded = df[uuid_logic]                                                  # excluding empty uuid to not exponential duplicates
            df_curr = df[~uuid_logic]                                                          # actual df without empty uuid 
        else: df_curr = df

        duplicate_log = df_curr.duplicated(subset=log_i,keep=False)                            # main logic to get the duplicates        
        all_duplicates = df_curr[duplicate_log].sort_values(by=log_i)                          # sort to group the duplicate cases
        
        if all_duplicates.empty: continue
        duplicates_inds = list(all_duplicates.index)                                      # the index of the duplicates will be drop from the main df

        all_duplicates['key'] = all_duplicates[log_i].apply('|'.join, axis=1).values      # the key column is the concated of the values of the keys
        all_duplicates['indexes'] = duplicates_inds                                       # saving the original index value for the merge
        all_duplicates['Logic'] = logic_name                                               
        
        all_duplicates = all_duplicates.sort_values(by='posting_date')                # sorting to extract the first case of each duplicate sorted by posting date
        unique_dup = all_duplicates.drop_duplicates(log_i, keep='first')              # unique duplicates by keys, the first dup row must be the early date

        def duplicates_calc(all_duplicates, unique_dup):
            if len(all_duplicates) == 0: return all_duplicates                            # if havent duplicates return the empty df 
            unique_dup['ID'] = [int(i) for i in range(countID, countID+len(unique_dup))]  # assigned each ID number to the each case 
            original_IDs = list(unique_dup['indexes']) 
            unique_dup = unique_dup[['key', 'ID']]
            all_d = pd.merge(all_duplicates,unique_dup,on='key',how='left')               # merging the cases with the original duplicate set by key
            all_d.loc[all_d['indexes'].isin(original_IDs), 'status'] = 'Original'         # the index of uniques dup have the tag Original
            all_d.loc[all_d['status']!='Original', 'status'] = 'Duplicated'               # the rest of the set have the Duplicated tag
            return all_d
        duplicates = duplicates_calc(all_duplicates, unique_dup)

        if duplicates.empty: continue
        
        dups_id = duplicates[duplicates['status']=='Duplicated']['indexes']
        df_curr = df_curr.drop(dups_id, axis=0)                                             # eliminating the ID used by duplicates
        if log_i == ['uuid']: df = pd.concat([df_excluded, df_curr])                        # adding the excluded empty uuid at the begin
        countID += len(unique_dup)                                                          # the next ID range will be the next value of the last duplicate shape
        duplicate_out = pd.concat([duplicate_out, duplicates])                              # concating to the main output df
        
    return duplicate_out

In [158]:
def get_duplicates(actual_day, all_data, historical, logics):

  print('running duplicates function...')
  print('initial shape', all_data.shape)
  out_cols = ['bsak', 'clrng_doc', 'document_date', 'currency', 'reference', 'ty',
              'dc_ind', 'amount', 'text', 'cocd', 'document_number', 'posting_date',
              'account', 'account_name', 'society', 'fiscal_year', 'user',
              'account_group', 'due_date', 'user_name', 'vendor', 'name_1',
              'account_ok', 'amount(w/cents)', 'key', 'reference(w/words)',
              'reference_words', 'uuid', 'tagged', 'id', 'status', 'key', 'indexes',
              'logic', 'indexes_h', 'state', 'indexes_d', 'account_f', 'carrier_key',
              'fbl1n']
  
  all_duplicates = pd.DataFrame(columns=all_data.columns)        # output dataframe
  df_data = copy.deepcopy(all_data)

  if historical:                                                  # get dups from all data and return
    duplicates = get_duplicates_hist(df_data, logics)             # the historical function calculate the duplicates in the same data set
    if duplicates.empty: return duplicates
    duplicates = duplicates.sort_values(by='ID')           
    duplicates = duplicates.reset_index(drop=True)
    all_duplicates = pd.concat([all_duplicates, duplicates])
    print('duplicates historical founded', all_duplicates.shape)
    print('\n')
    return all_duplicates

  date_format = r'%Y-%m-%d'
  df_data['posting_date']  = pd.to_datetime(df_data['posting_date'],  format=date_format, errors='coerce')                                       # 
  df_data['document_date'] = pd.to_datetime(df_data['document_date'], format=date_format, errors='coerce')          
  df_data['due_date']      = pd.to_datetime(df_data['due_date'],      format=date_format, errors='coerce')          

  daily_set = df_data[df_data['posting_date'] == actual_day]                       # data from current date
  histo_set = df_data[df_data['posting_date'] < actual_day]                        # data to the past from the curr date


  daily_set = daily_set.reset_index(drop=True)
  histo_set = histo_set.reset_index(drop=True)
  last_ID = 1
  histo_set = histo_set.reset_index(drop=True)
  duplicates_hist = get_duplicates_hist(daily_set, logics)                        # the historical function calculate the duplicates in the same set. only duplicates in daily set
  duplicates_hist["tipo_dup"] = "daily"

  if not duplicates_hist.empty:                                                   # if the result is not empty
    duplicates_hist = duplicates_hist.sort_values(by='ID')
    print('daily duplicates completed... \n', duplicates_hist.shape)
    all_duplicates = pd.concat([all_duplicates, duplicates_hist])                 # concat to the df the duplicates results
    special_dups = all_duplicates[all_duplicates['Logic']!='uuid']           # duplicates different to UUID

    dups = duplicates_hist[duplicates_hist['status']=='Duplicated']
    index_cal = list(dups['indexes'])                                        # indexes found of duplicates
    daily_set = daily_set[~(daily_set.index.isin(index_cal))]                # the new set is the data without dups found
    last_ID = duplicates_hist.ID.max()                                            # the new ID is the last of the dups found

  daily_set = daily_set.reset_index(drop=True)      
  duplicates_daily = get_duplicates_daily(histo_set, daily_set, last_ID, logics)   # the rest will be compare againts the historical
  duplicates_daily["tipo_dup"] = "historical"

  duplicates = pd.concat([duplicates_hist, duplicates_daily])

  if not duplicates.empty: 
    duplicates = duplicates.sort_values(by='ID')
    print('all duplicates completed... \n', duplicates.shape)
    all_duplicates = pd.concat([all_duplicates, duplicates])

  if 'ID' not in all_duplicates.columns: all_duplicates[['ID', 'Logic']] = '' 
  all_duplicates = all_duplicates.reset_index(drop=True)  
  print('final shape...', all_duplicates.shape)
  return all_duplicates

In [159]:
def reassing_ID(data):
  # ----------------------- re-assign ID                              
  data['ID'] = data['ID'].astype(str)
  data['ID'] = 'id_'+data['ID']                                          # add the string 'id_' turn the values in unique, the conversion can duplicate IDs
  old_IDs = list(data['ID'].unique())
  new_IDs = list(range(1, len(old_IDs)+1))
  all_IDs = list(zip(old_IDs, new_IDs))
  
  for ID_i in all_IDs: data['ID'] = data['ID'].replace(ID_i[0], ID_i[1])
  data['ID'] = data['ID'].astype(int)
  
  return data

# ------------------------------------------------------------------------------------------------------------------------------------
def reassing_status(df):
  if df.empty: return df
  final_df = pd.DataFrame()
  IDs_u = df['ID'].unique()                                                          # all ID of the set
  
  df['posting_date'] = pd.to_datetime(df['posting_date'])

  for ID in IDs_u:
    sub_df = df[df['ID']==ID].sort_values(by='posting_date', ascending=True)           # order to have the early first
    sub_df['posting_date'] = pd.to_datetime(sub_df['posting_date'])

    sub_df = sub_df.reset_index(drop=True)                                           # reset index to always have 0 index in sub df
    sub_df.loc[0, 'status'] = 'Original'                                             # reassing status
    sub_df.loc[1:, 'status'] = 'Duplicate'
    final_df = pd.concat([final_df, sub_df])                                         # concat all the sub dataframes
    final_df = final_df.reset_index(drop=True)                                         # reset all index
  
  return final_df

def format_output(data, Manual=False):
  out_columns = ['ID', 'status', 'Logic', 'Society', 'Account', 'user_name', 'Reference', 'reference_w_words', 'Amount',
                'amount_w_cents', 'Currency', 'Dollar_Amount', 'posting_date', 'clearing_document', 'document_date', 'item_text', 'uuid',
                'document_number', 'fiscal_year', 'users', 'due_date', 'duplicate_yes_no', 'comments']
  data = data.reset_index(drop=True)
  
  # ----------------------- Drop ID with UUID different
  #if not data.empty: data = filter_uuid(data)
  
  # ----------------------- Extra columns
  data['duplicate_yes_no'] = ''
  data['comments'] = ''
  
  # ----------------------- Dollar conversion
  data['amount_document_currency'] = pd.to_numeric(data['amount_document_currency'])
  data['Dollar_Amount'] = copy.deepcopy(data['amount_document_currency'])
  data['local_currency'] = data['local_currency'].astype(str).str.strip()
  
  # Prefer dataset FX for DOP when available. Keep static fallback for EUR.
  if 'exchange_rate' in data.columns:
    data['exchange_rate'] = pd.to_numeric(data['exchange_rate'], errors='coerce')
    dop_fx_logic = (data['local_currency']=='DOP') & (data['exchange_rate'] > 0)
    data.loc[dop_fx_logic, 'Dollar_Amount'] = data.loc[dop_fx_logic, 'amount_document_currency'] / data.loc[dop_fx_logic, 'exchange_rate']
  
  eur_c = 0.92
  data.loc[data['local_currency']=='EUR', 'Dollar_Amount'] = data['amount_document_currency']/eur_c
  data.Dollar_Amount = data.Dollar_Amount.astype(float)
  data.Dollar_Amount = data.Dollar_Amount.round(2) 


  # ----------------------- 
  data = data.reset_index(drop=True)
  data = reassing_ID(data)
  data = reassing_status(data)
  # ----------------------- 


  if data.empty: data['status'] = ''
  data = data.sort_values(by=['ID', 'status'], ascending=[True, False])
  
  data['posting_date']  = pd.to_datetime(data['posting_date'])
  data['posting_date']  = data['posting_date'].dt.strftime(r'%d-%m-%Y')
  data['document_date'] = pd.to_datetime(data['document_date'])
  data['document_date'] = data['document_date'].dt.strftime(r'%d-%m-%Y')
  data['due_date']      = pd.to_datetime(data['due_date'])
  data['due_date']      = data['due_date'].dt.strftime(r'%d-%m-%Y')
  
  data = data.rename(columns={'company_code': 'Society', 'account_number':'Account', 'reference_number':'Reference', 'amount_document_currency':'Amount',
                              'local_currency':'Currency'})
  for c in out_columns:
    if c not in data.columns: data[c] = ''
  data = data[out_columns].reset_index(drop=True)

  return data

## ------------------------------------------------ PROCESSING ------------------------------------------------

In [160]:
# Date window: today and 5 years of historical data.
today_dt = dt.date.today()
five_years_back_dt = today_dt - dt.timedelta(days=365 * 5)

# Keep env override support for local reruns.
today_date = os.getenv("DOP_CURR_DATE", today_dt.strftime("%Y-%m-%d"))
historic_start_date = os.getenv("DOP_HISTORIC_START_DATE", five_years_back_dt.strftime("%Y-%m-%d"))

# Preserve downstream variables used by the notebook.
run_mode = globals().get("RUN_MODE", os.getenv("DOP_RUN_MODE", "daily").strip().lower())
if run_mode not in {"daily", "historic"}:
    raise ValueError(f"Invalid run_mode={run_mode}. Use 'daily' or 'historic'.")
historic = run_mode == "historic"
yesterday = today_date
actual_year = yesterday.split('-')[0]
actual_month = yesterday.split('-')[1]

print('today_date', today_date)
print('historic_start_date', historic_start_date)
print('actual_year', actual_year)
print('actual_month', actual_month)
print('yesterday', yesterday)
print('historic', historic)
print('run_mode', run_mode)

today_date 2026-03-23
historic_start_date 2021-03-24
actual_year 2026
actual_month 03
yesterday 2026-03-23
historic True
run_mode historic


In [161]:

# ------------------------------------------------------------------------ Dates configuration
Historical = bool(historic)                                            # single toggle alignment: historic mode runs full historical duplicate detection

logics_locals = [['uuid'],
          ['account_number','reference_w_words','document_date','amount_document_currency','currency_key','company_code'],
          ['account_number','reference_w_words','amount_document_currency','document_date','currency_key','company_code'],
          ['account_number','reference_number','amount_w_cents','document_date','currency_key','company_code'],  
          ['account_number','reference_number','amount_document_currency','document_date','currency_key', 'company_code'], 
          ['account_number','reference_w_words','amount_w_cents','currency_key','company_code']]

logics_ext = [['uuid'],
          ['account_number','reference_w_words','document_date','amount_document_currency','currency_key','company_code'],
          ['account_number','reference_w_words','amount_document_currency','document_date','currency_key','company_code'],
          ['account_number','reference_number','amount_w_cents','document_date','currency_key','company_code'],  
          ['account_number','reference_number','amount_document_currency','document_date','currency_key', 'company_code'], 
          ['account_number','reference_w_words','amount_w_cents','currency_key','company_code'],
          ['account_number','amount_document_currency','document_date','currency_key', 'company_code'],
          ['account_number','amount_w_cents','currency_key', 'company_code']]

In [162]:
lfa1_path = os.path.join(run_output_dir, "dop_lfa1_extract.csv")
payables_path = os.path.join(run_output_dir, "dop_account_payables_extract.csv")
refresh_extract = os.getenv("DOP_REFRESH_EXTRACT", "true").lower() == "true"
os.makedirs(run_output_dir, exist_ok=True)

if refresh_extract or (not os.path.exists(lfa1_path)):
    spark_lfa1 = run_table_to_pandas("brewdat_uc_aurora_prod.slv_maz_masterdata_s4h.lfa1")
    spark_lfa1.to_csv(lfa1_path, index=False)
else:
    spark_lfa1 = pd.read_csv(lfa1_path)
save_step_df(spark_lfa1, "01_lfa1_input")

if refresh_extract or (not os.path.exists(payables_path)):
    spark_fbl1n = run_table_to_pandas(
        "brewdat_uc_maz_prod.gld_maz_finance_account_payables.do_account_payables",
        date_col="posting_date",
    )
    spark_fbl1n.to_csv(payables_path, index=False)
else:
    spark_fbl1n = pd.read_csv(payables_path)
save_step_df(spark_fbl1n, "02_payables_raw_input")

print('input shape:', spark_fbl1n.shape)
print('refresh_extract:', refresh_extract)

# ------------------------------------------------------------------------------------ formatting inputs
if "document_type" in spark_fbl1n.columns:
    spark_fbl1n = spark_fbl1n[spark_fbl1n['document_type'].astype(str) == 'RE'].copy()
# ------------------------------------------------------------------------------------
save_step_df(spark_fbl1n, "03_payables_re_filtered")
print('all inputs loaded...', spark_fbl1n.shape)

saved step output: local_outputs\mode_runs\historic\step_outputs\03_01_lfa1_input.csv shape=(4415, 207)
saved step output: local_outputs\mode_runs\historic\step_outputs\04_02_payables_raw_input.csv shape=(6566, 82)
input shape: (6566, 82)
refresh_extract: True
saved step output: local_outputs\mode_runs\historic\step_outputs\05_03_payables_re_filtered.csv shape=(1881, 82)
all inputs loaded... (1881, 82)


In [163]:
# document_currency = currency_key
# amount = amount_document_currency
# reference_number = reference_document
# item_text = financial_line_text
# amount_local_currency = amount_lc

rename_map = {
    "vendor_account": "account_number",
    "document_currency": "currency_key",
    "amount": "amount_document_currency",
    "reference_document": "reference_number",
    "financial_line_text": "item_text",
    "amount_lc": "amount_local_currency",
}
spark_fbl1n = spark_fbl1n.rename(columns=rename_map)
save_step_df(spark_fbl1n, "04_columns_renamed")

saved step output: local_outputs\mode_runs\historic\step_outputs\06_04_columns_renamed.csv shape=(1881, 82)


In [164]:
df = spark_fbl1n.copy()

# Keep alphanumeric vendor accounts; only normalize to trimmed string.
df["account_number"] = df["account_number"].astype(str).str.strip()
df = df[df["account_number"].notna()]
df = df[df["account_number"].str.len() > 0]

df["amount_document_currency"] = df["amount_document_currency"].astype(str)

spec_char_pattern = r"[&,;.\"\'#*()\[\]+/ ]"
df["reference_number"] = df["reference_number"].astype(str).str.strip().str.replace(spec_char_pattern, "", regex=True)

split_ref = df["reference_number"].str.split("-", n=1).str[0].str.strip()
mask_split = df["reference_number"].str.contains("-", regex=False, na=False) & (split_ref.str.len() == 8)
df.loc[mask_split, "reference_number"] = split_ref[mask_split]

df["Key"] = df["document_number"].astype(str) + df["company_code"].astype(str) + df["fiscal_year"].astype(str)
df["amount_w_cents"] = pd.to_numeric(df["amount_document_currency"], errors="coerce").fillna(0).astype(int).astype(str)

temp_digits = df["reference_number"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
ref_num = pd.to_numeric(temp_digits.replace("", np.nan), errors="coerce")
df["reference_w_words"] = ref_num.fillna(0).astype(int).astype(str)

df["reference_words"] = df["reference_number"].astype(str).str.replace(r"[^a-zA-Z]", "", regex=True)

first_part_text = df["item_text"].astype(str).str.split(" - ", n=1).str[0].str.strip()
dashes_count = first_part_text.str.count("-")
uuid_mask = (first_part_text.str.len() == 36) & (dashes_count == 4)
df["uuid"] = ""
df.loc[uuid_mask, "uuid"] = first_part_text[uuid_mask].str.upper()

# Output final
spark_fbl1n = df.copy()
save_step_df(spark_fbl1n, "05_preprocessed_payables")

saved step output: local_outputs\mode_runs\historic\step_outputs\07_05_preprocessed_payables.csv shape=(1881, 87)


In [165]:
spark_fbl1n = spark_fbl1n.copy()
spark_fbl1n["posting_date"] = pd.to_datetime(spark_fbl1n["posting_date"], errors="coerce")
spark_fbl1n["document_date"] = pd.to_datetime(spark_fbl1n["document_date"], errors="coerce")

if historic or Historical:
    # In historic mode, keep the full extraction window instead of limiting to fiscal-year logic.
    full_window_mask = (
        (spark_fbl1n["posting_date"] >= pd.to_datetime(historic_start_date, errors="coerce"))
        & (spark_fbl1n["posting_date"] <= pd.to_datetime(today_date, errors="coerce"))
    )
    spark_df_0 = spark_fbl1n[full_window_mask].copy()
    print('Historic Window Filter', spark_df_0.shape[0])
else:
    year_i = int(actual_year)
    posting_years = spark_fbl1n["posting_date"].dropna().dt.year
    if (posting_years == year_i).sum() == 0 and not posting_years.empty:
        fallback_year = int(posting_years.max())
        print(f"No posting rows for actual_year={year_i}; using latest available posting year={fallback_year}")
        year_i = fallback_year

    cond1 = spark_fbl1n["posting_date"].dt.year == year_i
    cond2 = spark_fbl1n["document_date"].dt.year == year_i
    cond3 = spark_fbl1n["document_date"].dt.year == year_i - 1
    cond4 = spark_fbl1n["posting_date"].dt.year == year_i - 1

    # ------- logic to query
    if (actual_month == "01") or (actual_month == "02"):
        logic_spark = (cond1 & cond2) | (cond1 & cond3) | (cond3 & cond4)
    else:
        logic_spark = (cond1 & cond2) | (cond1 & cond3)

    spark_df_0 = spark_fbl1n[logic_spark].copy()
    print('Fiscal Year Filter', spark_df_0.shape[0])

save_step_df(spark_df_0, "06_window_filtered")

Historic Window Filter 1881
saved step output: local_outputs\mode_runs\historic\step_outputs\08_06_window_filtered.csv shape=(1881, 87)


In [166]:
spark_df_1 = spark_df_0.copy()
spark_df_1["account_number"] = spark_df_1["account_number"].astype(str).str.replace(r'^[0]*', '', regex=True)  # drop initial 0
spark_df_1 = spark_df_1[spark_df_1["account_number"].str.len() > 4]  # no intercompany

spark_df_1["clearing_document"] = spark_df_1["clearing_document"].astype(str)
spark_df_1["account_number"] = spark_df_1["account_number"].astype(str)
spark_df_1["clearing_document"] = spark_df_1["clearing_document"].str.replace(r'^[0]*', '', regex=True)  # drop initial 0

spark_df_1 = spark_df_1[~spark_df_1["clearing_document"].str.startswith("14", na=False)]

save_step_df(spark_df_1, "07_business_filtered")
print('filter bussiness rule shape', spark_df_1.shape)

saved step output: local_outputs\mode_runs\historic\step_outputs\09_07_business_filtered.csv shape=(1881, 87)
filter bussiness rule shape (1881, 87)


In [167]:
pandas_fbl1n = spark_df_1.copy()
save_step_df(pandas_fbl1n, "08_pandas_input")

saved step output: local_outputs\mode_runs\historic\step_outputs\10_08_pandas_input.csv shape=(1881, 87)


In [168]:
pandas_fbl1n_locals = pandas_fbl1n[pandas_fbl1n["vendor_country_code"]=="DO"]
pandas_fbl1n_ext = pandas_fbl1n[pandas_fbl1n["vendor_country_code"]!="DO"]
save_step_df(pandas_fbl1n_locals, "09_locals_split")
save_step_df(pandas_fbl1n_ext, "10_external_split")

saved step output: local_outputs\mode_runs\historic\step_outputs\11_09_locals_split.csv shape=(1421, 87)
saved step output: local_outputs\mode_runs\historic\step_outputs\12_10_external_split.csv shape=(460, 87)


In [169]:
all_duplicates_locals = get_duplicates(yesterday, pandas_fbl1n_locals, Historical, logics_locals)    # find duplicates in the data formated
save_step_df(all_duplicates_locals, "11_duplicates_locals")

running duplicates function...
initial shape (1421, 87)
duplicates historical founded (30, 92)


saved step output: local_outputs\mode_runs\historic\step_outputs\13_11_duplicates_locals.csv shape=(30, 92)


In [170]:
all_duplicates_ext = get_duplicates(yesterday, pandas_fbl1n_ext, Historical, logics_ext)    # find duplicates in the data formated
save_step_df(all_duplicates_ext, "12_duplicates_external")

running duplicates function...
initial shape (460, 87)
duplicates historical founded (696, 92)


saved step output: local_outputs\mode_runs\historic\step_outputs\14_12_duplicates_external.csv shape=(696, 92)


In [171]:
all_duplicates = pd.concat([all_duplicates_locals, all_duplicates_ext])
save_step_df(all_duplicates, "13_duplicates_combined")

saved step output: local_outputs\mode_runs\historic\step_outputs\15_13_duplicates_combined.csv shape=(726, 92)


In [172]:
print('initial shape', all_duplicates.shape)

df = all_duplicates

if df.empty:
    real_duplicates = df.copy()
else:
    df_orig = spark_df_0.copy()

    uuid_logic = df['Logic']=='uuid'
    uuid_excluded = df[uuid_logic]                      # bussiness rule: not apply to the logic UUID
    df_to_filter = df[~uuid_logic]

    df_account = copy.deepcopy(df_to_filter)

    # ----------------------------------------------------------------- formating accounts from duplicates
    df_account['indexes'] = list(df_account.index)
    df_account['reference_w_words'] = df_account['reference_w_words'].astype(str).str.replace(r'[^0-9]', '', regex=True).replace('', '0')
    df_account['company_code'] = df_account['company_code'].astype(str)
    df_account['account_number'] = df_account['account_number'].astype(str)
    df_account['account_number'] = df_account['account_number'].str.lstrip('0')
    df_account['cross_ref_key'] = df_account['account_number'] + df_account['company_code'] + df_account['reference_w_words']
    df_account['fbl1n'] = True                                                                                 # key for the merge

    uniq_accounts = list(df_account['account_number'].unique())
    uniq_socs = list(df_account['company_code'].unique())

    # ----------------------------------------------------------------- Filtering historical data with duplicates data
    df_orig['account_number'] = df_orig['account_number'].astype(str)
    df_orig['account_number'] = df_orig['account_number'].str.lstrip('0')
    df_orig['company_code'] = df_orig['company_code'].astype(str)
    df_cond1 = df_orig.document_type.astype(str).str.startswith('RE', na=False)
    df_cond2 = df_orig.account_number.isin(uniq_accounts)
    df_cond3 = df_orig.company_code.isin(uniq_socs)

    histo_accounts = df_orig[(df_cond1) & (df_cond2) & (df_cond3)]

    # ----------------------------------------------------------------- formating accounts from historical
    spec_char_list=['&',',',';','.','"','\'','#','*','(',')','[',']','+','/'," "]
    for char in spec_char_list:
        histo_accounts['reference_number'] = histo_accounts['reference_number'].astype(str).apply(lambda x: x.strip().replace(char, ''))

    try:
        histo_accounts_refs = histo_accounts[histo_accounts['reference_number'].str.contains('-', na=False)]
        histo_accounts_refs['Sub_Reference'] = histo_accounts_refs['reference_number'].str.split('-', n=1, expand=True)[0]  # splitting by the firm part of the '-'
        reference_logic = histo_accounts_refs.Sub_Reference.str.len()==8                                          # only reference in the format that have the len 8
        histo_accounts_refs = histo_accounts_refs[reference_logic]
        correct_sub_ref = histo_accounts_refs['Sub_Reference']                                                    # get the reference with the correct format
        histo_accounts.loc[histo_accounts.index.isin(correct_sub_ref.index), 'reference_number'] = list(correct_sub_ref)
    except Exception:
        pass

    histo_accounts["reference_w_words"] = histo_accounts["reference_number"].apply(lambda x: "".join(re.findall(r'(\d+(?:\.\d+)?)', x)))
    histo_accounts["reference_w_words"] = histo_accounts["reference_w_words"].apply(lambda x: x.lstrip("0"))
    histo_accounts['reference_w_words'] = histo_accounts['reference_w_words'].astype(str).str.replace(r'[^0-9]', '', regex=True).replace('', '0')
    histo_accounts['company_code'] = histo_accounts['company_code'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    histo_accounts['account_number'] = histo_accounts['account_number'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip().str.lstrip('0')

    histo_accounts['cross_ref_key'] = histo_accounts['account_number'] + histo_accounts['company_code'] + histo_accounts['reference_w_words']

    # ----------------------------------------------------------------- looking for the key duplicates on the keys historical
    dup_account = pd.merge(histo_accounts, df_account, on='cross_ref_key', how='left')

    fbl1n_logic = dup_account['fbl1n'] == True                                                     # the keys from duplicates found in historical fbl1n
    dup_accounts_T = dup_account[fbl1n_logic]

    accounts_count = dup_accounts_T.groupby('cross_ref_key').agg({'ID':'first', 'cross_ref_key':'count'}).reset_index(drop=True)    # count the carrier key occurrences
    logic_carriers = accounts_count['cross_ref_key'] >= 2
    index_dup = accounts_count[logic_carriers]                                                     # only get the count more or equal to 2

    ID_dup = list(index_dup['ID'].unique())
    accounts_dup = df_account[df_account.ID.isin(ID_dup)]                                          # extract the ID found from the original df

    real_duplicates = pd.concat([uuid_excluded, accounts_dup])

save_step_df(real_duplicates, "14_real_duplicates")
print('final shape', real_duplicates.shape)
print('\n')

initial shape (726, 92)
saved step output: local_outputs\mode_runs\historic\step_outputs\16_14_real_duplicates.csv shape=(348, 94)
final shape (348, 94)




In [173]:
if "tipo_dup" not in real_duplicates.columns:
    real_duplicates['tipo_dup'] = np.nan
else:
    real_duplicates.sort_values(by = ["document_number", "tipo_dup"], ascending = [True, True])
    real_duplicates.drop_duplicates(subset = ["document_number"], inplace = True, keep = "last")
    print('current shape', real_duplicates.shape)

In [174]:
# Manual vs automated vendor split removed for DOP (no manual vendor CSV available).
df = format_output(real_duplicates)
save_step_df(df, "15_formatted_output")
print('shape after format:', df.shape)

saved step output: local_outputs\mode_runs\historic\step_outputs\17_15_formatted_output.csv shape=(348, 23)
shape after format: (348, 23)


In [175]:
# Manual/automated branch removed; dataset is processed as a single flow.
print('single flow shape: ', df.shape)

single flow shape:  (348, 23)


In [176]:
df['validation'] = 'Automated'

if df.empty:
  print('empty file')
  if 'comments' not in df.columns:
    df['comments'] = ''
  df.loc[0, 'comments'] = 'No duplicate payments were found'
else:
  # Re-assign IDs only when the raw duplicate schema is present.
  if 'ID' in df.columns:
    df = reassing_ID(df)

df.columns = [i.lower() for i in df.columns]
df['report_date'] = yesterday
df = df.reset_index(drop=True)
save_step_df(df, "16_validated_output")

saved step output: local_outputs\mode_runs\historic\step_outputs\18_16_validated_output.csv shape=(348, 25)


In [177]:
if historic: df['monthly'] = 1
else:        df['monthly'] = 0

df['document_date'] = pd.to_datetime(df['document_date'], format=r'%d-%m-%Y', errors='coerce').dt.strftime(r'%d/%m/%Y')
df['posting_date'] = pd.to_datetime(df['posting_date'], format=r'%d-%m-%Y', errors='coerce').dt.strftime(r'%d/%m/%Y')
# Accept both ISO and already-formatted dates during reruns.
df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce', dayfirst=True).dt.strftime(r'%d/%m/%Y')
df['due_date'] = pd.to_datetime(df['due_date'], format=r'%d-%m-%Y', errors='coerce').dt.strftime(r'%d/%m/%Y')

for c in df.columns: df[c] = df[c].astype(str)
save_step_df(df, "17_dates_normalized_output")

saved step output: local_outputs\mode_runs\historic\step_outputs\19_17_dates_normalized_output.csv shape=(348, 26)


In [178]:
# Local-only persistence: append with existing local report and save CSV (no cloud writes).
import os
import re
import calendar

month_i = int(yesterday.split('-')[1])
month = calendar.month_name[month_i].lower()
year = actual_year

if Historical:
    outname = 'do_duplicates_report_historical.csv'
elif historic:
    outname = f'do_duplicates_report_historic_{month}_{year}.csv'
else:
    outname = f'do_duplicates_report_{month}_{year}.csv'

os.makedirs(run_output_dir, exist_ok=True)
out_path = os.path.join(run_output_dir, outname)

# Normalize legacy Spanish columns to English.
legacy_col_map = {
    'duplicado_si_no': 'duplicate_yes_no',
    'comentarios': 'comments',
}

def normalize_legacy_columns(frame):
    frame = frame.rename(columns=legacy_col_map)
    frame.columns = [
        re.sub(r'^comments\.\d+$', 'comments', re.sub(r'^duplicate_yes_no\.\d+$', 'duplicate_yes_no', c))
        for c in frame.columns
    ]
    frame = frame.loc[:, ~frame.columns.duplicated()].copy()
    return frame

df = normalize_legacy_columns(df)

append_existing = os.getenv('DOP_APPEND_EXISTING', 'false').lower() == 'true'
if append_existing and os.path.exists(out_path):
    df_prev = pd.read_csv(out_path)
    df_prev = normalize_legacy_columns(df_prev)
    df = pd.concat([df_prev, df], ignore_index=True)

# Keep output clean and deterministic.
df = df.replace('nan', '')
df = df.fillna('')
df = df.drop_duplicates()
if 'report_date' in df.columns:
    df = df.sort_values(by='report_date', ascending=True)

df = normalize_legacy_columns(df)
df = df.reset_index(drop=True)
for c in df.columns:
    df[c] = df[c].astype(str)

save_step_df(df, "18_final_dataset_before_save")
df.to_csv(out_path, index=False)
print('file saved at...', out_path)
print('final output shape', df.shape)

saved step output: local_outputs\mode_runs\historic\step_outputs\20_18_final_dataset_before_save.csv shape=(348, 26)
file saved at... local_outputs\mode_runs\historic\do_duplicates_report_historical.csv
final output shape (348, 26)


In [179]:
# months_parse = {1:'enero', 2:'febrero', 3:'marzo', 4:'abril', 5:'mayo', 6:'junio', 7:'julio', 8:'agosto', 9:'septiembre', 10:'octubre', 11:'noviembre', 12:'diciembre'}
# path_base = "/Volumes/brewdat_uc_mazana_dev/manual_maz_cts/cts/finance/ptp/duplicate_payments/do/output/"
# month_i = int(yesterday.split('-')[1])
# month = months_parse[month_i]
# year = actual_year

# if Historical:  outname = f'do_duplicates_report_historical.csv'
# elif historic:  outname = f'do_duplicates_report_historic_{month}_{year}.csv'
# else:           outname = f'do_duplicates_report_{month}_{year}.csv'

# try:
#     df_i = pd.read_csv(path_base+outname)
#     df = pd.concat([df, df_i])    
# except Exception as Error: pass

# df = df.replace('nan', '')
# df = df.fillna('')
# df = df.drop_duplicates()
# df = df.sort_values(by='report_date', ascending=True)

# df = df.reset_index(drop=True)
# for c in df.columns: df[c] = df[c].astype(str)
# df.to_csv(path_base+outname, index=False)

# print('file saved at...', path_base+outname)